# Anime_DeadFramesRemover(DFR)
**Created by [xyether](https://github.com/XYETHER).**

Select a **T4 GPU** runtime, run DFR Setup, then Remove dead frames. DFR exports a **shortened, silent, 23.976 FPS** video; it does not preserve input timing. The optional RIFE cells interpolate either your input or the DFR output.

Depth selects attention regions; original pixels decide whether a frame is kept. Type 1 analyses every third frame; Type 2 every second frame. Review fast motion, blinks and scene transitions.

[Instructions and examples](https://github.com/XYETHER/Anime_DeadFramesRemover-DFR).


In [ ]:
#@title 1. DFR — Setup
import os,sys,subprocess,pathlib,urllib.request,hashlib,importlib.util
DFR_HOME=pathlib.Path('/content/DFR');DFR_HOME.mkdir(parents=True,exist_ok=True)
import torch
if not torch.cuda.is_available():raise RuntimeError('DFR requires a GPU runtime. Select a T4 GPU and rerun.')
missing=[package for module,package in [('gdown','gdown'),('tqdm','tqdm'),('cv2','opencv-python-headless'),('PIL','pillow')] if importlib.util.find_spec(module) is None]
if missing:subprocess.run([sys.executable,'-m','pip','install','-q',*missing],check=True)
if not __import__('shutil').which('ffmpeg'):
    subprocess.run(['apt-get','update','-qq'],check=True)
    subprocess.run(['apt-get','install','-y','-qq','ffmpeg'],check=True)
DFR_REV='a561b849ebae10a6f5ef49e26c83cbbcd36c71bf'
DFR_REPO=DFR_HOME/('depth-anything-v2-'+DFR_REV[:10])
if not (DFR_REPO/'depth_anything_v2'/'dpt.py').exists():
    import zipfile,io,shutil
    with urllib.request.urlopen('https://github.com/DepthAnything/Depth-Anything-V2/archive/'+DFR_REV+'.zip',timeout=120) as response:
        archive=zipfile.ZipFile(io.BytesIO(response.read()))
    # Only copy this pinned official repository; no archive paths are trusted.
    prefix='Depth-Anything-V2-'+DFR_REV+'/'
    for item in archive.infolist():
        if not item.filename.startswith(prefix) or item.is_dir():continue
        relative=pathlib.PurePosixPath(item.filename[len(prefix):])
        if '..' in relative.parts or relative.is_absolute():raise RuntimeError('Unsafe repository archive path')
        destination=DFR_REPO.joinpath(*relative.parts);destination.parent.mkdir(parents=True,exist_ok=True)
        destination.write_bytes(archive.read(item))
DFR_CHECKPOINT=DFR_HOME/'depth_anything_v2_vits.pth'
DFR_WEIGHTS_REV='03876f8651c73a60fe4c2c48294e09fcb6838fcf'
expected='715fade13be8f229f8a70cc02066f656f2423a59effd0579197bbf57860e1378'
if not DFR_CHECKPOINT.exists():
    temp=DFR_CHECKPOINT.with_suffix('.partial')
    try:
        urllib.request.urlretrieve('https://huggingface.co/depth-anything/Depth-Anything-V2-Small/resolve/'+DFR_WEIGHTS_REV+'/depth_anything_v2_vits.pth',temp)
        if hashlib.sha256(temp.read_bytes()).hexdigest()!=expected:raise RuntimeError('Incomplete or corrupt model download; rerun setup.')
        temp.replace(DFR_CHECKPOINT)
    finally:
        if temp.exists():temp.unlink()
expected='715fade13be8f229f8a70cc02066f656f2423a59effd0579197bbf57860e1378'
if hashlib.sha256(DFR_CHECKPOINT.read_bytes()).hexdigest()!=expected:
    raise RuntimeError('Cached model checksum mismatch. Delete /content/DFR/depth_anything_v2_vits.pth and rerun setup.')
if str(DFR_REPO) not in sys.path:sys.path.insert(0,str(DFR_REPO))
print('DFR ready |',torch.cuda.get_device_name(0),'| Depth Anything V2 Small | cached setup')


In [ ]:
#@title 2. DFR — Remove dead frames
"""DFR decision core: depth is attention, original pixels are evidence."""
import cv2
import numpy as np

def foreground(depth, focus=50):
    """Relative inverse depth: larger=nearer. None signals unusable depth."""
    if depth is None or not np.isfinite(depth).all(): return None
    lo,hi=np.percentile(depth,[2,98])
    if hi-lo < max(1e-6,abs(float(np.median(depth)))*.005): return None
    d=np.uint8(np.clip((depth-lo)/(hi-lo),0,1)*255)
    threshold,_=cv2.threshold(d,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    threshold=float(np.clip(threshold+(focus-50)*1.4,25,225))
    mask=np.uint8(d>threshold)
    scale=max(1.,max(depth.shape)/640)
    close_size=int(round(7*scale))|1;erode_size=int(round(3*scale))|1
    mask=cv2.morphologyEx(mask,cv2.MORPH_CLOSE,np.ones((close_size,close_size),np.uint8))
    count,labels,stats,_=cv2.connectedComponentsWithStats(mask,8)
    clean=np.zeros_like(mask)
    for i in range(1,count):
        if stats[i,cv2.CC_STAT_AREA]>=max(9,mask.size*.00005): clean[labels==i]=1
    clean=cv2.erode(clean,np.ones((erode_size,erode_size),np.uint8))
    if clean.mean()<.025 or clean.mean()>.96:return None
    return clean.astype(bool)

def prepare(frame):
    return cv2.GaussianBlur(frame.astype(np.float32),(3,3),.65)

def difference(a,b,mask,strength=50):
    """Higher strength tolerates more change and removes more frames."""
    if mask is None:mask=np.ones(a.shape[:2],bool)
    diff=np.max(np.abs(a-b),axis=2)
    # A local patch gate protects small eye/mouth motion from global dilution.
    pixel=2.0+strength*.08
    active=(diff>pixel)&mask
    area=max(1,int(mask.sum()))
    fraction=float(active.sum()/area)
    # Keep the spatial meaning of sensitivity consistent at 640px and 1080p.
    scale=max(1.,max(mask.shape)/640);large_size=round(20*scale);tiny_size=round(8*scale)
    mass=cv2.boxFilter(mask.astype(np.float32),-1,(large_size,large_size),normalize=False)
    changed=cv2.boxFilter(active.astype(np.float32),-1,(large_size,large_size),normalize=False)
    valid=mass>large_size*large_size*.4
    local=float(np.max(changed[valid]/mass[valid])) if valid.any() else 0.
    global_limit=.004+strength*.00020
    local_limit=.08+strength*.0024
    # Separate high-contrast small-detail gate: a brief blink must not be
    # diluted by a 20x20 window or wait to accumulate over several frames.
    tiny=((diff>max(12.,pixel*2))&mask).astype(np.float32)
    tiny_mass=cv2.boxFilter(mask.astype(np.float32),-1,(tiny_size,tiny_size),normalize=False)
    tiny_count=cv2.boxFilter(tiny,-1,(tiny_size,tiny_size),normalize=False)
    valid_tiny=tiny_mass>=tiny_size*tiny_size*.5
    tiny_peak=float(np.max(tiny_count[valid_tiny]/tiny_mass[valid_tiny])) if valid_tiny.any() else 0.
    score=max(fraction/global_limit,local/local_limit,tiny_peak/(.10+strength*.001))
    return float(score),fraction,local

def is_cut(a,b):
    """Only strong, widespread discontinuities count; motion handles others."""
    x=cv2.resize(a,(96,54));y=cv2.resize(b,(96,54))
    d=np.mean(np.abs(x-y),axis=2)
    return bool(np.mean(d)>48 and np.mean(d>25)>.70)

class Selector:
    def __init__(self,strength=50):
        self.strength=strength;self.anchor=None;self.prev=None;self.mask=None;self.index=-1
    def step(self,frame,mask,index):
        f=prepare(frame);cut=self.prev is not None and is_cut(self.prev,f)
        reason='first' if self.anchor is None else ('cut' if cut else '')
        unreliable=mask is None or self.mask is None
        region=None if unreliable else mask|self.mask
        score,frac,local=(0.,0.,0.) if self.anchor is None else difference(self.anchor,f,region,self.strength)
        # Flat/unreliable depth falls back to stricter full-frame comparison.
        if unreliable and self.anchor is not None:
            score,frac,local=difference(self.anchor,f,None,min(self.strength,25))
        keep=bool(reason or score>=1)
        anchor_index=self.index
        if keep:self.anchor=f;self.mask=mask;self.index=index
        self.prev=f
        return {'frame':index,'keep':keep,'reference':anchor_index,'reason':reason or ('motion' if keep else 'held'),'score':score,'changed_fraction':frac,'local_change':local,'mask_area':float(mask.mean()) if mask is not None else 1.,'fallback':unreliable}

def self_test():
    rng=np.random.default_rng(7);a=np.zeros((180,320,3),np.uint8)+50
    a[40:160,100:230]=150;mask=np.zeros((180,320),bool);mask[45:155,105:225]=1
    z=Selector();assert z.step(a,mask,0)['keep'];assert not z.step(a,mask,1)['keep']
    b=a.copy();b[:,:70]=230;assert not z.step(b,mask,2)['keep'],'background-only'
    b=a.copy();b[80:84,150:168]=10;assert z.step(b,mask,3)['keep'],'eye motion'
    z=Selector();z.step(a,mask,0);noise=np.clip(a.astype(float)+rng.normal(0,.6,a.shape),0,255).astype('uint8');assert not z.step(noise,mask,1)['keep'],'codec-like noise'
    z=Selector();z.step(a,mask,0)
    found=False
    for i in range(1,21):
        b=a.copy();b[70:100,140:180]+=i
        if z.step(b,mask,i)['keep']:found=True;break
    assert found,'slow cumulative change lost'
    for hh,ww in [(1,8),(2,12),(3,12)]:
        z=Selector();z.step(a,mask,0);blink=a.copy();blink[80:80+hh,150:150+ww]=0
        assert z.step(blink,mask,1)['keep'],('tiny blink',hh,ww)
        assert z.step(a,mask,2)['keep'],'return from blink'
    assert foreground(np.ones((180,320))) is None
    z=Selector();z.step(a,mask,0);assert z.step(255-a,mask,1)['keep'],'cut lost'
    for strength in [0,25,50,75,100]:
        score,*_=difference(prepare(a),prepare(b),mask,strength)
        if strength:assert score<=last+1e-6
        last=score
    print('PASS: duplicate, independent background, eye, noise, cumulative change, flat depth, cut, sensitivity')


import pathlib,time,json,csv,subprocess,os

def analyse_video(path,depth_fn,strength=50,focus=50,output_dir='/content/DFR/results',save_cache=False,analysis_width=640,frame_step=1):
    path=pathlib.Path(path);out=pathlib.Path(output_dir);out.mkdir(parents=True,exist_ok=True)
    if frame_step not in (1,2,3):raise ValueError("Invalid frame step")
    timing=json.loads(subprocess.check_output(['ffprobe','-v','error','-select_streams','v:0','-show_entries','frame=best_effort_timestamp_time','-of','json',str(path)]))
    timestamps=[float(f['best_effort_timestamp_time']) for f in timing.get('frames',[]) if 'best_effort_timestamp_time' in f]
    cap=cv2.VideoCapture(str(path))
    if not cap.isOpened():raise ValueError('Cannot decode '+str(path))
    w,h=int(cap.get(3)),int(cap.get(4));total=int(cap.get(7))
    scale=min(1.,analysis_width/max(w,h));aw,ah=max(2,round(w*scale)),max(2,round(h*scale))
    sel=Selector(strength);kept_count=0;rows=[];shots=[];cached=[];last_frame=None;mask=None;depth=None;depth_calls=0;t=time.perf_counter()
    stage={k:0. for k in ['decode_s','resize_s','depth_s','foreground_s','decision_s']}
    try:
        while True:
            stamp=time.perf_counter();i=len(rows);ok=cap.grab()
            if not ok:break
            if i%frame_step:
                stage['decode_s']+=time.perf_counter()-stamp
                row=dict(rows[-1]);row.update(frame=i,keep=False,reason='pre_skip',output_frame='',score=0.,changed_fraction=0.,local_change=0.,mask_area=0.,fallback=False,source_time_s=timestamps[i] if i<len(timestamps) else float(cap.get(cv2.CAP_PROP_POS_MSEC)/1000))
                rows.append(row);continue
            ok,f=cap.retrieve()
            if not ok:raise RuntimeError('Frame retrieval failed')
            stage['decode_s']+=time.perf_counter()-stamp
            stamp=time.perf_counter();small=cv2.resize(f,(aw,ah),interpolation=cv2.INTER_AREA);i=len(rows);stage['resize_s']+=time.perf_counter()-stamp
            # Exact low-resolution duplicates need no repeated network inference.
            if last_frame is None or not np.array_equal(small,last_frame):
                stamp=time.perf_counter();depth=depth_fn(small);stage['depth_s']+=time.perf_counter()-stamp
                stamp=time.perf_counter();mask=foreground(depth,focus);stage['foreground_s']+=time.perf_counter()-stamp;depth_calls+=1
            stamp=time.perf_counter();row=sel.step(small,mask,i);stage['decision_s']+=time.perf_counter()-stamp
            row['source_time_s']=timestamps[i] if i<len(timestamps) else float(cap.get(cv2.CAP_PROP_POS_MSEC)/1000)
            row['output_frame']=kept_count if row['keep'] else ''
            kept_count+=int(row['keep'])
            rows.append(row)
            if save_cache:cached.append((small.copy(),depth.copy()))
            if i in set([0,1,max(1,total//3),max(1,total*2//3),max(1,total-2)]):
                shots.append((i,small.copy(),None if mask is None else mask.copy()))
            last_frame=small
            if i%100==0: print(path.name,i,'/',total,flush=True)
    finally:cap.release()
    if not rows:raise ValueError('No frames decoded')
    if timestamps and len(rows)!=len(timestamps):raise RuntimeError('Decoder/frame timestamp count mismatch; refusing partial output')
    # Decoder frame count is the ground truth; metadata can be wrong on VFR.
    analysis_seconds=time.perf_counter()-t
    csvpath=out/(path.stem+'_DFR_decisions.csv')
    with csvpath.open('w',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)
    from PIL import Image,ImageDraw
    sheet=Image.new('RGB',(aw*2,(ah+24)*len(shots)),(20,20,20));draw=ImageDraw.Draw(sheet)
    for j,(i,im,m) in enumerate(shots):
        rgb=cv2.cvtColor(im,cv2.COLOR_BGR2RGB);vis=rgb.copy()
        if m is not None:vis[~m]=(vis[~m]*.16).astype('uint8')
        sheet.paste(Image.fromarray(rgb),(0,j*(ah+24)+24));sheet.paste(Image.fromarray(vis),(aw,j*(ah+24)+24))
        draw.text((4,j*(ah+24)+5),f'Frame {i} | original / attention region (NOT output) | '+rows[i]['reason'],fill='white')
    sheet.save(out/(path.stem+'_DFR_attention.jpg'))
    summary={'name':path.name,'input_frames':len(rows),'kept_frames':sum(r['keep'] for r in rows),'analysis_seconds':analysis_seconds,'analysis_fps':len(rows)/analysis_seconds,'depth_calls':depth_calls,'fallback_frames':sum(r['fallback'] for r in rows),'vfr_detected':bool(len(timestamps)>2 and np.ptp(np.diff(timestamps))>.001),'cuts':[r['frame'] for r in rows if r['reason']=='cut'],'strength':strength,'focus':focus,'width':w,'height':h}
    summary['stage_seconds']=stage;summary['analysis_size']=[aw,ah]
    summary['frame_step']=frame_step;summary['analysed_frames']=sum(r['reason']!='pre_skip' for r in rows)
    (out/(path.stem+'_DFR_summary.json')).write_text(json.dumps(summary,indent=2))
    if save_cache:
        np.savez_compressed(out/(path.stem+'_cache.npz'),frames=np.stack([v[0] for v in cached]),depths=np.stack([v[1] for v in cached]).astype(np.float16))
    print(json.dumps(summary),flush=True)
    return rows,summary

def encode_video(path,rows,output_dir='/content/DFR/results',fps_override=0,encoder='h264_nvenc'):
    from fractions import Fraction
    path=pathlib.Path(path);out=pathlib.Path(output_dir);out.mkdir(parents=True,exist_ok=True)
    probe=json.loads(subprocess.check_output(['ffprobe','-v','error','-select_streams','v:0','-show_streams','-of','json',str(path)]))['streams'][0]
    rate=str(fps_override) if fps_override else probe.get('avg_frame_rate','0/0')
    if rate in ['0/0','0']:rate=probe.get('r_frame_rate','24/1')
    fps=float(Fraction(rate));assert 0<fps<=240,rate
    cap=cv2.VideoCapture(str(path));w,h=int(cap.get(3)),int(cap.get(4))
    final=out/(path.stem+'_DFR.mp4');temp=out/(path.stem+'_DFR.partial.mp4')
    cmd=['ffmpeg','-hide_banner','-loglevel','error','-y','-f','rawvideo','-pix_fmt','bgr24','-s',f'{w}x{h}','-framerate',rate,'-i','pipe:0','-an']
    if encoder not in ['h264_nvenc','hevc_nvenc']:
        raise ValueError('DFR supports NVIDIA hardware encoding only: H.264 or H.265')
    cmd += ['-c:v',encoder,'-preset','p7','-tune','hq','-rc','vbr','-cq','22','-b:v','0',
            '-multipass','fullres','-rc-lookahead','32','-spatial-aq','1','-temporal-aq','1',
            '-aq-strength','8','-bf','3','-b_ref_mode','middle','-g','240']
    if encoder=='hevc_nvenc':cmd += ['-tag:v','hvc1']
    cmd += ['-pix_fmt','yuv420p' if w%2==0 and h%2==0 else 'yuv444p','-movflags','+faststart']
    for flag,key in [('-color_primaries','color_primaries'),('-color_trc','color_transfer'),('-colorspace','color_space')]:
        if probe.get(key) not in [None,'unknown','reserved','unspecified']:cmd += [flag,probe[key]]
    # Explicit RGB-to-YUV matrix; merely copying colour tags does not
    # configure conversion and can miscolour HD output.
    matrix='bt709' if probe.get('color_space','bt709') in ['bt709','unknown'] else 'bt601'
    if probe.get('color_space') in [None,'unknown','unspecified']:
        cmd += ['-colorspace','bt709','-color_primaries','bt709','-color_trc','bt709']
    cmd += ['-vf','scale=in_range=full:out_range=tv:out_color_matrix='+matrix,'-color_range','tv',str(temp)]
    start=time.perf_counter();n=0;i=0;log=out/(path.stem+'_encode.log')
    try:
        with log.open('wb') as stderr:
            enc=subprocess.Popen(cmd,stdin=subprocess.PIPE,stdout=subprocess.DEVNULL,stderr=stderr)
            try:
                while True:
                    ok=cap.grab()
                    if not ok:break
                    if i>=len(rows):raise RuntimeError('Input changed between analysis and encoding')
                    if rows[i]['keep']:
                        ok,f=cap.retrieve()
                        if not ok:raise RuntimeError('Frame retrieval failed')
                        enc.stdin.write(f.tobytes());n+=1
                    i+=1
                enc.stdin.close();ret=enc.wait(timeout=180)
                if ret:raise RuntimeError('Hardware encoder failed. No CPU fallback. '+log.read_text())
            except BaseException:
                enc.kill();enc.wait();raise
        if i!=len(rows):raise RuntimeError('Decode ended before all analyzed frames')
        check=json.loads(subprocess.check_output(['ffprobe','-v','error','-count_frames','-show_streams','-of','json',str(temp)]))
        v=next(s for s in check['streams'] if s['codec_type']=='video')
        assert int(v['nb_read_frames'])==n and int(v['width'])==w and int(v['height'])==h,check
        assert Fraction(v['avg_frame_rate'])==Fraction(rate),v
        assert v['codec_name']==('hevc' if encoder=='hevc_nvenc' else 'h264'),v
        assert all(s['codec_type']!='audio' for s in check['streams'])
        temp.replace(final)
        result={'output':str(final),'output_frames':n,'output_fps':rate,'output_duration':n/fps,'encode_seconds':time.perf_counter()-start,'encoder':encoder,'verified':True}
        print(json.dumps(result),flush=True);return result
    finally:
        cap.release()
        if temp.exists():temp.unlink()

# Processing controls and launch — appended after self-contained core.
import torch,sys,urllib.parse,urllib.request,uuid,zipfile,shutil
from IPython.display import display,Video,Image as DisplayImage,FileLink

def dfr_inputs(source):
    from google.colab import files
    incoming=pathlib.Path('/content/DFR/input');incoming.mkdir(parents=True,exist_ok=True)
    if not source.strip():
        uploaded=files.upload()
        if not uploaded:raise ValueError('No video uploaded')
        paths=[]
        for name,data in uploaded.items():
            p=incoming/pathlib.Path(name).name
            if p.exists() and p.read_bytes()!=data:p=incoming/(p.stem+'_'+uuid.uuid4().hex[:6]+p.suffix)
            p.write_bytes(data);paths.append(p)
        return paths
    parsed=urllib.parse.urlparse(source.strip())
    if parsed.scheme in ['https','http']:
        if parsed.hostname not in ['drive.google.com','docs.google.com']:
            raise ValueError('Use a public Google Drive file link, a /content path, or leave blank to upload.')
        import gdown
        p=incoming/('video_'+uuid.uuid4().hex[:8]+'.mp4')
        for attempt in range(3):
            try:
                got=gdown.download(url=source.strip(),output=str(p),quiet=True,fuzzy=True)
                if not got or not p.exists() or p.stat().st_size==0:raise RuntimeError('Public Drive download failed')
                return [p]
            except Exception:
                if attempt==2:raise
                time.sleep(3)
    p=pathlib.Path(source.strip())
    if p.is_dir():
        paths=sorted(x for x in p.iterdir() if x.suffix.lower() in ['.mp4','.mkv','.mov','.webm','.avi'] and not x.stem.endswith('_DFR'))
        if not paths:raise ValueError('No source videos in this folder')
        return paths
    if not p.is_file():raise ValueError('Video path does not exist: '+str(p))
    return [p]

def dfr_run(source='',codec='H.264 (NVIDIA)',auto_download=True,processing_type='Type 1'):
    strength,focus,quality,analysis=70,50,'Balanced','Fast (640px)'
    global DFR_MODEL
    from fractions import Fraction
    step={"Type 1":3,"Type 2":2}[processing_type]
    if not 0<=strength<=100 or not 0<=focus<=100:raise ValueError('Sliders must be between 0 and 100')
    size={'Fast':196,'Balanced':252,'Detailed':392}[quality]
    encoder={'H.264 (NVIDIA)':'h264_nvenc','H.265 (NVIDIA)':'hevc_nvenc'}[codec]
    width={'Fast (640px)':640,'Full 1080p':1920}[analysis]
    if not torch.cuda.is_available():raise RuntimeError('Select a GPU runtime, then run Setup.')
    if 'DFR_CHECKPOINT' not in globals() or not DFR_CHECKPOINT.exists():raise RuntimeError('Run the Setup cell first.')
    inputs=dfr_inputs(source)
    if len({p.stem.casefold() for p in inputs})!=len(inputs):
        raise ValueError('Input filenames share a stem. Rename them before batch processing to avoid output collisions.')
    # Real NVENC probe at source dimensions; fail explicitly, never use CPU.
    for p in inputs:
        stream=json.loads(subprocess.check_output(['ffprobe','-v','error','-select_streams','v:0','-show_streams','-of','json',str(p)]))['streams'][0]
        if stream.get('color_transfer') in ['smpte2084','arib-std-b67']:raise ValueError('DFR currently supports SDR anime, not HDR sources.')
        w,h=int(stream['width']),int(stream['height'])
        probe=subprocess.run(['ffmpeg','-hide_banner','-loglevel','error','-f','lavfi','-i',f'color=s={w}x{h}:r=24','-frames:v','1','-c:v',encoder,'-f','null','-'],capture_output=True,text=True,timeout=30)
        if probe.returncode:raise RuntimeError('NVIDIA hardware encoder unavailable. No CPU fallback. '+probe.stderr)
    torch.set_num_threads(2);cv2.setNumThreads(2)
    if 'DFR_MODEL' not in globals():
        from depth_anything_v2.dpt import DepthAnythingV2
        DFR_MODEL=DepthAnythingV2(encoder='vits',features=64,out_channels=[48,96,192,384])
        DFR_MODEL.load_state_dict(torch.load(DFR_CHECKPOINT,map_location='cpu',weights_only=True))
        DFR_MODEL=DFR_MODEL.eval().cuda()
    @torch.inference_mode()
    def get_depth(frame):
        with torch.autocast('cuda',dtype=torch.float16):
            tensor,_=DFR_MODEL.image2tensor(frame,size);depth=DFR_MODEL(tensor)[0]
        depth=cv2.resize(depth.float().cpu().numpy(),(frame.shape[1],frame.shape[0]))
        if not np.isfinite(depth).all():raise RuntimeError('Depth model returned invalid values')
        return depth
    out=pathlib.Path('/content/DFR/output')/uuid.uuid4().hex[:8];out.mkdir(parents=True)
    batch=[]
    for p in inputs:
        start=time.perf_counter()
        rows,summary=analyse_video(p,get_depth,strength,focus,out,analysis_width=width,frame_step=step)
        source_stream=json.loads(subprocess.check_output(['ffprobe','-v','error','-select_streams','v:0','-show_streams','-of','json',str(p)]))['streams'][0]
        rate=Fraction(24000,1001)
        summary.update(encode_video(p,rows,out,encoder=encoder,fps_override=str(rate)))
        summary['processing_type']=processing_type
        summary['total_seconds']=time.perf_counter()-start
        summary['full_pipeline_fps']=summary['input_frames']/summary['total_seconds']
        summary['depth_quality']=quality;summary['depth_model']='Depth Anything V2 Small'
        batch.append(summary)
        (out/'DFR_report.json').write_text(json.dumps(batch,indent=2))
        print(f"{p.name}: {summary['input_frames']} → {summary['kept_frames']} frames | {summary['full_pipeline_fps']:.1f} full-pipeline FPS | {summary['total_seconds']:.1f}s")
        if summary['vfr_detected']:print('Variable-rate input: output is fixed at 23.976 FPS; original timestamps are in the CSV.')
        if len(inputs)==1:
            display(DisplayImage(filename=str(out/(p.stem+'_DFR_attention.jpg')),width=800))
            if encoder=='h264_nvenc':display(Video(summary['output'],embed=True,width=800))
            else:print('H.265 browser playback varies. Download the MP4 to preview in your editor.')
    print('Done. Original colour/resolution; shortened silent video. No interpolation. Results:',out)
    for result in batch:
        display(FileLink(str(result['output'])))
    if auto_download:
        from google.colab import files
        for result in batch:
            files.download(str(result['output']))
    return batch

#@markdown **Input:** public Google Drive file link or `/content` file/folder. Blank = upload. No Drive mount.
Video_source = "" #@param {type:"string"}
#@markdown Type 1 is more reliable though might delete some frames, while type 2 is is less sensitive but might keep some deadframes (type 1 is reccomended).
Processing_type = "Type 1" #@param ["Type 1", "Type 2"]
Codec = "H.265 (NVIDIA)" #@param ["H.264 (NVIDIA)", "H.265 (NVIDIA)"]
Auto_download = True #@param {type:"boolean"}
DFR_results = dfr_run(Video_source,Codec,Auto_download,Processing_type)


# **INTERPOLATION (Bonus) - RIFE**

In [ ]:
#@title #**SETUP**

import os
import subprocess

Auto_Reconnect = True #@param {type:"boolean"}

RIFE_DOWNLOADS = {
    "v4.18": {"id": "1I9qOQ9NoKxHBVC0hHwF1LnqWL_GANCs5", "zip": "RIFE_v4.18.zip", "target_dir": "RIFE_v4.18"},
    "v4.22": {"id": "1q34qP2IW6QlizMhblld9h5lah2VF3_l6", "zip": "RIFE_v4.22.zip", "target_dir": "RIFE_v4.22"},
    "v4.26": {"id": "1gViYvvQrtETBgU1w8axZSsr7YUuw31uy", "zip": "RIFE_v4.26.zip", "target_dir": "RIFE_v4.26"},
}

RIFE_DIR = "/content/Practical-RIFE"

print("⏳ Installing dependencies...")
!pip install -q gdown
!apt-get install -y -qq ffmpeg &> /dev/null

print("⏳ Cloning Practical-RIFE...")
if not os.path.exists(RIFE_DIR):
    !git clone --depth 1 https://github.com/hzwer/Practical-RIFE.git {RIFE_DIR} &> /dev/null
# Restore the pinned upstream script before patching, so setup is repeatable.
RIFE_REV = "bbfd2ea90910789a860ea3e2b32a240cd577b75e"
subprocess.run(["git", "-C", RIFE_DIR, "fetch", "--depth", "1", "origin", RIFE_REV], check=True, stdout=subprocess.DEVNULL)
_pristine = subprocess.check_output(["git", "-C", RIFE_DIR, "show", RIFE_REV + ":inference_video.py"], text=True)
with open(os.path.join(RIFE_DIR, "inference_video.py"), "w") as _f:
    _f.write("\n".join(line for line in _pristine.splitlines() if "import moviepy.editor" not in line) + "\n")

RIFE_SCRIPT = os.path.join(RIFE_DIR, "inference_video.py")
with open(RIFE_SCRIPT) as _f:
    _src = _f.read()

_src = _src.replace("import _thread", "import _thread\nimport subprocess")

_writer = "    vid_out = cv2.VideoWriter(vid_out_name, fourcc, args.fps, (w, h))"
_pipe = '''    _enc = os.environ.get("RIFE_ENCODER", "h264_nvenc")
    vid_out_proc = subprocess.Popen(["ffmpeg", "-y", "-f", "rawvideo", "-vcodec", "rawvideo",
                                     "-s", "%dx%d" % (w, h), "-pix_fmt", "bgr24", "-r", str(args.fps), "-i", "-",
                                     "-c:v", _enc, "-preset", "p7", "-tune", "hq", "-rc", "vbr", "-cq", "17", "-b:v", "0",
                                     "-pix_fmt", "yuv420p", vid_out_name, "-hide_banner", "-loglevel", "error"],
                                    stdin=subprocess.PIPE)
    vid_out = vid_out_proc'''
assert _writer in _src, "writer anchor missing"
_src = _src.replace(_writer, _pipe)

_write = "            vid_out.write(item[:, :, ::-1])"
_write_new = "            vid_out_proc.stdin.write(item[:, :, ::-1].tobytes())"
assert _write in _src, "write anchor missing"
_src = _src.replace(_write, _write_new)

_release = "if not vid_out is None:\n    vid_out.release()"
_release_new = "if not vid_out is None:\n    vid_out_proc.stdin.close()\n    vid_out_proc.wait()"
assert _release in _src, "release anchor missing"
_src = _src.replace(_release, _release_new)

# Join the writer before closing its pipe; an empty queue can still be writing.
_src = _src.replace("import _thread", "import _thread\nimport threading")
_thread_anchor = "_thread.start_new_thread(clear_write_buffer, (args, write_buffer))"
assert _thread_anchor in _src, "writer thread anchor missing"
_src = _src.replace(_thread_anchor, "_writer_thread = threading.Thread(target=clear_write_buffer, args=(args, write_buffer))\n_writer_thread.start()")
_wait_anchor = "while(not write_buffer.empty()):\n    time.sleep(0.1)"
assert _wait_anchor in _src, "writer wait anchor missing"
_src = _src.replace(_wait_anchor, "_writer_thread.join()")


_rd_imp = "import skvideo.io\n"
assert _rd_imp in _src, "skvideo import anchor missing"
_src = _src.replace(_rd_imp, "")

_vreader = "    videogen = skvideo.io.vreader(args.video)"
_reader = '''    _cap2 = cv2.VideoCapture(args.video)
    _vw = int(_cap2.get(cv2.CAP_PROP_FRAME_WIDTH)); _vh = int(_cap2.get(cv2.CAP_PROP_FRAME_HEIGHT))
    _cap2.release()
    _rd = subprocess.Popen(["ffmpeg", "-hwaccel", "cuda", "-v", "error", "-i", args.video, "-f", "rawvideo", "-pix_fmt", "rgb24", "-"],
                           stdout=subprocess.PIPE, bufsize=10**8)
    def _vgen():
        while True:
            raw = _rd.stdout.read(_vw * _vh * 3)
            if not raw or len(raw) < _vw * _vh * 3:
                break
            yield np.frombuffer(raw, dtype=np.uint8).reshape(_vh, _vw, 3)
    videogen = _vgen()'''
assert _vreader in _src, "vreader anchor missing"
_src = _src.replace(_vreader, _reader)

_audio_call = "if args.png == False and fpsNotAssigned == True and not args.video is None:"
assert _audio_call in _src, "audio anchor missing"
_src = _src.replace(_audio_call, "if False and args.png == False and fpsNotAssigned == True and not args.video is None:")

with open(RIFE_SCRIPT, "w") as _f:
    _f.write(_src)
print("✅ inference_video.py patched: ffmpeg reader + NVENC writer (cq 17), skvideo & moviepy out of the loop")

print("⏳ Downloading RIFE models...")
import gdown, time
for name, info in RIFE_DOWNLOADS.items():
    zip_path = os.path.join(RIFE_DIR, info["zip"])
    target_dir = os.path.join(RIFE_DIR, info["target_dir"])
    if os.path.exists(os.path.join(target_dir, "train_log")):
        print(f"  {name}: already present")
        continue
    for attempt in range(3):
        gdown.download(id=info["id"], output=zip_path, quiet=True, fuzzy=True)
        if os.path.exists(zip_path) and os.path.getsize(zip_path) > 1000000:
            break
        time.sleep(3)
    if not os.path.exists(zip_path) or os.path.getsize(zip_path) < 1000000:
        raise RuntimeError(f"❌ {name} download failed (Drive quota?)")
    os.makedirs(target_dir, exist_ok=True)
    !unzip -o -q "{zip_path}" -d "{target_dir}"
    os.remove(zip_path)
    print(f"  {name}: OK")

if Auto_Reconnect:
    from IPython.display import display, Javascript
    display(Javascript('''
    function ClickConnect(){
      btn = document.querySelector("colab-connect-button");
      if (btn != null) btn.click();
      btn = document.getElementById('ok');
      if (btn != null) btn.click();
    }
    setInterval(ClickConnect,60000)
    '''))

print("\n✅ Setup complete. Ready to process.")

In [ ]:
#@title #**PROCESS VIDEO**
# ==================================
# INTERPOLATION ONLY (RIFE)
# ==================================
import os, sys, subprocess, shutil

RIFE_MODEL_DIR = "/content/Practical-RIFE"
RIFE_MODELS = {
    "v4.26": "RIFE_v4.26",
    "v4.22": "RIFE_v4.22",
    "v4.18": "RIFE_v4.18",
}
DEADFRAMES_FPS = {
    "Off": None,
    "2s (11.98 fps)": 11.98,
    "3s (7.992 fps)": 7.992,
    "4s (5.994 fps)": 5.994,
}
SLOWMO_FPS = {
    "Off": None,
    "60 fps": 60,
    "30 fps": 30,
    "24 fps": 24,
}

def remove_dead_frames(video_path, fps_target):
    """Drop frames to a fraction of 23.976 (2s/3s/4s holds) via ffmpeg fps filter (NVENC encode)."""
    encoder = os.environ.get("RIFE_ENCODER", "h264_nvenc")
    out = os.path.join(os.path.dirname(video_path), "deadframes_" + os.path.basename(video_path))
    cmd = ["ffmpeg", "-y", "-hwaccel", "cuda", "-i", video_path, "-vf", f"fps={fps_target}",
           "-c:v", encoder, "-preset", "p7", "-tune", "hq", "-rc", "vbr", "-cq", "17", "-b:v", "0",
           "-c:a", "copy", "-map", "0", out,
           "-hide_banner", "-loglevel", "error"]
    subprocess.run(cmd, check=True)
    print(f"✅ Dead frames removed -> {os.path.basename(out)}")
    return out

def has_audio(video_path):
    r = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "a",
                       "-show_entries", "stream=index", "-of", "csv=p=0", video_path],
                      capture_output=True, text=True)
    return r.stdout.strip() != ""

def interpolate_video(video_path, model_folder, multi, slowmo_fps):
    model_version_path = os.path.join(RIFE_MODEL_DIR, model_folder)
    model_tlog_path = os.path.join(model_version_path, "train_log")
    if not os.path.exists(model_tlog_path):
        raise FileNotFoundError(f"❌ train_log not found in {model_version_path}. Run SETUP first.")

    target = int(multi)
    if target > 8:
        stages = []
        m = target
        while m > 1:
            if m % 2 == 0:
                stages.append(2); m //= 2
            else:
                stages.append(m); m = 1
    else:
        stages = [target]

    current = video_path
    n_stages = len(stages)
    for i, s in enumerate(stages):
        is_last = (i == n_stages - 1)
        out = f"/content/interp_stage_{i}.mp4"
        env = dict(os.environ, PYTHONPATH=model_version_path)
        cmd = [sys.executable, "/content/Practical-RIFE/inference_video.py",
               "--model", model_tlog_path, "--multi", str(s), "--fp16",
               "--video", current, "--output", out]
        if slowmo_fps and is_last:
            cmd += ["--fps", str(slowmo_fps)]
        print(f"⏳ Stage {i+1}/{n_stages}: {s}x (total {target}x, encoder: {os.environ.get('RIFE_ENCODER', 'h264_nvenc')})...")
        subprocess.run(cmd, check=True, env=env)
        if current != video_path and os.path.exists(current):
            os.remove(current)
        current = out

    if slowmo_fps is None and has_audio(video_path):
        final = os.path.splitext(current)[0] + "_audio.mp4"
        subprocess.run(["ffmpeg", "-y", "-i", current, "-i", video_path,
                       "-map", "0:v:0", "-map", "1:a:0?", "-c", "copy", final,
                       "-hide_banner", "-loglevel", "error"], check=True)
        os.replace(final, current)
    print(f"✅ Interpolation complete -> {os.path.basename(current)}")
    return current

def process_single_video(video_path, interpolation_factor, rife_model_folder, deadframes_label, slowmo_label):
    output_dir = "/content/interpolated"
    os.makedirs(output_dir, exist_ok=True)
    current = video_path

    if DEADFRAMES_FPS[deadframes_label]:
        current = remove_dead_frames(current, DEADFRAMES_FPS[deadframes_label])
    current = interpolate_video(current, rife_model_folder, int(interpolation_factor), SLOWMO_FPS[slowmo_label])

    suffix = f"interpolated-{interpolation_factor}x"
    if slowmo_label != "Off":
        suffix += f"-slowmo{SLOWMO_FPS[slowmo_label]}"
    base, ext = os.path.splitext(os.path.basename(video_path))
    final = os.path.join(output_dir, f"{base}-{suffix}{ext}")
    if os.path.exists(final):
        os.remove(final)
    shutil.move(current, final)
    print(f"🏁 Finished: {final}")
    return final

# ------------------- Processing params -------------------
video_path = "/content/video.mp4" #@param {type:"string"}
interpolation_factor = 8 #@param {type:"string"}  # typed by user, default 8
rife_model = "v4.26" #@param ["v4.26", "v4.22", "v4.18"]
deadframes = "Off" #@param ["Off", "2s (11.98 fps)", "3s (7.992 fps)", "4s (5.994 fps)"]
slow_down = "Off" #@param ["Off", "60 fps", "30 fps", "24 fps"]
encoder = "h264_nvenc" #@param ["h264_nvenc", "hevc_nvenc"]
auto_download = True #@param {type:"boolean"}

model_folder = RIFE_MODELS[rife_model]
os.environ["RIFE_ENCODER"] = encoder
try:
    interpolation_factor = int(str(interpolation_factor).strip())
except ValueError:
    raise ValueError(f"❌ Interpolation factor must be a whole number (e.g. 8, 16), got: {interpolation_factor!r}")

if os.path.isdir(video_path):
    exts = (".mp4", ".mov", ".avi", ".mkv", ".webm")
    files_list = [os.path.join(video_path, f) for f in os.listdir(video_path) if f.lower().endswith(exts)]
    print(f"📁 Batch: {len(files_list)} videos")
    for f in files_list:
        try:
            process_single_video(f, interpolation_factor, model_folder, deadframes, slow_down)
        except Exception as e:
            print(f"❌ {os.path.basename(f)} failed: {e}")
    print("\n✅ Batch complete.")
else:
    if not os.path.exists(video_path):
        raise FileNotFoundError(f"❌ Video not found: {video_path}")
    final = process_single_video(video_path, interpolation_factor, model_folder, deadframes, slow_down)
    if auto_download:
        from google.colab import files
        files.download(final)